In [ ]:
## Setup
import sys
import os
import json
import re

# Dodaje folder nadrzędny do ścieżki aby widzieć folder 'backend'
sys.path.append(os.path.abspath(os.path.join('..')))

from backend.rag_engine import LaborLawRAG

# 1. Ładowanie pytań testowych
with open('test_questions.json', 'r', encoding='utf-8') as f:
    test_questions = json.load(f)

# 2. Inicjalizacja silnika
rag = LaborLawRAG()

# 3. Funkcja do normalizacji
def normalize_art(art_name):
    # Usuwa wszystko co nie jest literą lub cyfrą (np. nawiasy, kropki, spacje)
    return re.sub(r'[^a-z0-9]', '', str(art_name).lower())

In [ ]:
def evaluate(rag_instance, test_data, limit=20):
    hits = {5: 0, 10: 0, 20: 0}
    mrr_sum = 0
    total = len(test_data)

    print(f"Testowanie {total} pytań...")

    for item in test_data:
        query = item['question']
        expected = normalize_art(item['expected_art'])
        
        # Pobiera tylko kontekst
        _, sources = rag_instance.get_context(query, limit=limit)
        normalized_sources = [normalize_art(s) for s in sources]

        # Oblicza Recall@K
        for k in [5, 10, 20]:
            if expected in normalized_sources[:k]:
                hits[k] += 1
        
        # Oblicza MRR
        if expected in normalized_sources:
            rank = normalized_sources.index(expected) + 1
            mrr_sum += 1 / rank

    return {
        "R@5": round(hits[5]/total, 3),
        "R@10": round(hits[10]/total, 3),
        "R@20": round(hits[20]/total, 3),
        "MRR": round(mrr_sum/total, 3)
    }

In [ ]:
results = {}
# Sprawdza różne proporcje: 0.0 (tylko słowa) -> 1.0 (tylko sens)
alphas = [0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]

for a in alphas:
    rag.alpha = a
    score = evaluate(rag, test_questions)
    results[a] = score
    print(f"Alpha {a}: {score}")